# Workshop 01: Time Series Graphics & Diagnostics

**Course:** Machine Learning II: Forecasting and Dynamic Modeling  
**Institution:** Lazarski University  
**Instructor:** Leonardo Di Caterina  
**Duration:** 120 Minutes (Laboratory Session)  
**Reference:** Hyndman & Athanasopoulos, *FPP3*, Chapters 1–2  

---

## Executive Overview
In time series analysis, visual diagnostics are not decorative — **the geometry of the data dictates the econometric model class.**

### 120-Minute Laboratory Map
1. **Part 1 (25m):** Temporal Data Ingestion — `DatetimeIndex`, frequency enforcement (`asfreq`), and safe data validation.
2. **Part 2 (30m):** Visual Anatomy — Time plots, seasonal plots, and seasonal subseries plots on US Retail Sales.
3. **Part 3 (20m):** Lag Scatter Grids — Visualizing serial dependence geometry before summarising it with a single number.
4. **Part 4 (25m):** Autocorrelation (ACF) & White Noise — Sample ACF, Bartlett bounds, and the Ljung-Box portmanteau test.
5. **Part 5 (20m):** Interview Knowledge Checks — Answering the 3 syllabus practical questions & Git repository hygiene.

In [ ]:
# Environment & Library Imports
from typing import Dict, List, Any, Union, Tuple, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

# Visual aesthetics & reproducibility
sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Resolve data directory relative to this notebook
DATA_DIR = Path('..') / 'data'
print(f'Data directory: {DATA_DIR.resolve()}')
print('Available datasets:', [f.name for f in sorted(DATA_DIR.glob('*.csv'))])

## Part 1: Temporal Data Ingestion & Frequency Architecture (25 Minutes)

A standard integer `RangeIndex` destroys all temporal properties. A time series DataFrame **must** possess a monotonically sorted `pd.DatetimeIndex` with an explicit calendar frequency.

### Key Distinction
- **`asfreq('MS')`**: Imposes a regular monthly grid on existing timestamps, exposing missing intervals as `NaN`.
- **`resample('MS').mean()`**: Aggregates higher-frequency data into monthly summary bins.

### Task 1: `load_and_index_series`
**Function Signature:** `load_and_index_series(filepath: str, date_col: str, val_col: str, freq: str = 'MS') -> pd.Series`

#### 🛠️ Step-by-Step Logic to Write:
1. **Load Raw Data:** Use `pd.read_csv(filepath)` to read the CSV dataset into a DataFrame.
2. **Parse Dates:** Use `pd.to_datetime(df[date_col])` to convert the date string column into `datetime64[ns]` timestamps.
3. **Clean Numeric Values:** Use `pd.to_numeric(df[val_col], errors='coerce')` to cast the value column into floats (unparseable values become `NaN`).
4. **Set Datetime Index:** Use `df.set_index(date_col)` to establish timestamps as the DataFrame index.
5. **Sort Chronologically:** Use `.sort_index()` to guarantee monotonically increasing chronological order (oldest to newest).
6. **Extract Target Series:** Extract `df[val_col]` as a `pd.Series`.
7. **Enforce Frequency:** Call `.asfreq(freq)` on the Series to impose the calendar frequency grid (e.g., `'MS'` for Month Start).

#### 🧰 Recommended Functions & Methods:
- `pd.read_csv(filepath)`: Ingests tabular CSV data.
- `pd.to_datetime(...)`: Robustly parses date strings into pandas Timestamps.
- `pd.to_numeric(..., errors='coerce')`: Safely converts numeric series to float, converting parse errors to `NaN`.
- `df.set_index(...)`: Moves a column to become the DataFrame index.
- `df.sort_index()`: Enforces chronological monotonic order.
- `series.asfreq(freq)`: Enforces regular temporal grid without aggregation.

#### ⚠️ Critical Pitfalls to Avoid:
- **Do NOT use `.resample()` here**: We are enforcing a frequency grid on already-monthly series, not downsampling daily data.
- **Always call `.sort_index()` before `.asfreq()`**: Unsorted timestamps will cause `.asfreq()` to fail or produce corrupted date alignment.

In [ ]:
def load_and_index_series(
    filepath: str,
    date_col: str,
    val_col: str,
    freq: str = 'MS'
) -> pd.Series:
    """
    Ingests raw time series CSV, enforces an immutable DatetimeIndex and calendar frequency.

    Parameters
    ----------
    filepath : str
        Path to CSV file.
    date_col : str
        Name of the column containing date strings.
    val_col : str
        Name of the numeric target column.
    freq : str, default='MS'
        Pandas frequency alias (e.g., 'MS' for Month Start).

    Returns
    -------
    pd.Series
        Sorted, named Series with DatetimeIndex and explicit frequency.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 1
indpro = load_and_index_series(DATA_DIR / 'INDPRO.csv', 'observation_date', 'INDPRO')
retail = load_and_index_series(DATA_DIR / 'RSAFS.csv', 'observation_date', 'RSAFSNA')
retail_sa = load_and_index_series(DATA_DIR / 'RSAFS_SA.csv', 'observation_date', 'RSAFS')

print(f'INDPRO: {len(indpro)} obs, freq={indpro.index.freq}, range: {indpro.index[0].date()} to {indpro.index[-1].date()}')
print(f'Retail (unadj): {len(retail)} obs, freq={retail.index.freq}')
print(f'Retail (SA): {len(retail_sa)} obs, freq={retail_sa.index.freq}')
print(f'\nIndex is monotonic: {indpro.index.is_monotonic_increasing}')
print(f'Missing values in INDPRO: {indpro.isna().sum()}')

## Part 2: Visual Anatomy — Time, Seasonal & Subseries Plots (30 Minutes)

Three fundamental time series visualizations separate long-run **trend**, calendar **seasonality**, and macroeconomic **cycles**:
1. **Time Plot:** Full trajectory over time — reveals structural breaks, level shifts, and variance expansion.
2. **Seasonal Plot:** Overlays all years onto a common 12-month axis — reveals intra-year regularities (December retail peak).
3. **Seasonal Subseries Plot:** Groups all Januarys, all Februarys, etc., with horizontal mean bars — reveals whether specific months change structurally over time.

### Task 2: `extract_seasonal_components`
**Function Signature:** `extract_seasonal_components(series: pd.Series) -> pd.DataFrame`

#### 🛠️ Step-by-Step Logic to Write:
1. **Convert to DataFrame:** Convert the input Series to a DataFrame using `df = series.to_frame(name='val')`.
2. **Extract Year & Month:** Create two new columns from the `DatetimeIndex`: `df['year'] = series.index.year` and `df['month'] = series.index.month`.
3. **Pivot Table:** Use `.pivot(index='year', columns='month', values='val')` to reshape the long table into a wide Year x Month matrix.
4. **Tidy Names:** Ensure the index name is `'year'` (using `.rename_axis(index='year', columns=None)`).

#### 🧰 Recommended Functions & Methods:
- `series.index.year` and `series.index.month`: Fast datetime index accessors returning integer years and months (1 to 12).
- `series.to_frame(name='val')`: Converts a 1D Series into a 2D DataFrame for tabular reshaping.
- `df.pivot(index='year', columns='month', values='val')`: Reshapes data into an $N_{\text{years}} \times 12$ matrix.
- `df.rename_axis(index='year', columns=None)`: Strips extraneous column axis names and ensures the index is named `'year'`.

#### ⚠️ Critical Pitfalls to Avoid:
- **Keep Month columns as integers `1..12`**: Do not format columns as string names (`'Jan'`, `'Feb'`) — autograders and downstream plotting code expect integer keys `1..12`.
- **Index Name:** Ensure `table.index.name == 'year'`, as required by the autograding test suite.

In [ ]:
def extract_seasonal_components(series: pd.Series) -> pd.DataFrame:
    """
    Reshapes a monthly DatetimeIndex series into a Year x Month matrix.

    Parameters
    ----------
    series : pd.Series
        Monthly time series with DatetimeIndex.

    Returns
    -------
    pd.DataFrame
        Table with index = Years (int) and columns = Months (1..12).
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 2: Three-Panel Visual Diagnostic Dashboard

# Use last 15 years of retail data for clarity
retail_recent = retail['2008':]
seasonal_table = extract_seasonal_components(retail_recent)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Time Plot
axes[0].plot(retail_recent.index, retail_recent.values, color='#2563eb', lw=1.2)
axes[0].set_title('Time Plot: US Retail Sales (Unadjusted)', fontsize=11)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Millions ($)')

# Panel 2: Seasonal Plot (overlay years)
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(seasonal_table)))
for i, (yr, row) in enumerate(seasonal_table.iterrows()):
    axes[1].plot(range(1, 13), row.values, color=cmap[i], alpha=0.6, lw=0.8, label=str(yr) if i % 5 == 0 else '')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_labels, fontsize=8)
axes[1].set_title('Seasonal Plot (Yearly Overlay)', fontsize=11)
axes[1].set_xlabel('Month')
axes[1].legend(fontsize=7, loc='upper left')

# Panel 3: Seasonal Subseries Plot
for m in range(1, 13):
    month_vals = seasonal_table[m].dropna()
    x_positions = np.full(len(month_vals), m)
    axes[2].scatter(x_positions, month_vals.values, color='#2563eb', s=10, alpha=0.5)
    axes[2].hlines(month_vals.mean(), m - 0.3, m + 0.3, color='#dc2626', lw=2)
axes[2].set_xticks(range(1, 13))
axes[2].set_xticklabels(month_labels, fontsize=8)
axes[2].set_title('Seasonal Subseries Plot', fontsize=11)
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Millions ($)')

plt.tight_layout()
plt.show()

## Part 3: Dynamic Dependence & Lag Scatter Grids (20 Minutes)

In cross-sectional ML, observations are assumed independent. In time series, $y_t$ depends systematically on $y_{t-1}, y_{t-2}, \dots, y_{t-k}$.

Before summarizing this relationship with a scalar autocorrelation, inspect the **Lag Scatter Matrix**:
- Strong linear diagonal at lag 1 → high persistence.
- Tight clustering at lag 12 → annual seasonality.
- Diffuse circular cloud → near white noise.

### Task 3: `build_lag_matrix`
**Function Signature:** `build_lag_matrix(series: pd.Series, lags: List[int]) -> pd.DataFrame`

#### 🛠️ Step-by-Step Logic to Write:
1. **Initialize Table:** Create a dictionary or DataFrame with the contemporaneous series: `data = {'y_t': series}`.
2. **Iterate Over Lags:** For each integer lag order $k$ in `lags`:
   - Shift the series forward by $k$ time periods using `series.shift(k)`.
   - Store the shifted series in the dictionary under column name `f'lag_{k}'` (e.g. `'lag_1'`, `'lag_12'`).
3. **Return DataFrame:** Construct and return `pd.DataFrame(data)` with the original `DatetimeIndex` intact.

#### 🧰 Recommended Functions & Methods:
- `pd.DataFrame({'y_t': series})`: Initialize DataFrame with target.
- `series.shift(k)`: Lag operator $L^k y_t = y_{t-k}$. A positive integer $k$ shifts past values forward into current rows.
- Dictionary comprehension: `{'y_t': series, **{f'lag_{k}': series.shift(k) for k in lags}}`.

#### ⚠️ Critical Pitfalls to Avoid:
- **Shift Direction:** Always use a positive integer $k$ in `series.shift(k)`. A negative shift `series.shift(-k)` shifts future values into the present, introducing fatal lookahead bias / data leakage!
- **Preserve initial `NaN`s:** The first $k$ rows for lag $k$ will naturally be `NaN` because observations prior to the start of the series do not exist. Do not call `.dropna()` inside `build_lag_matrix`.

In [ ]:
def build_lag_matrix(series: pd.Series, lags: List[int]) -> pd.DataFrame:
    """
    Constructs a DataFrame containing the original series and specified lagged copies.

    Parameters
    ----------
    series : pd.Series
        Input time series.
    lags : List[int]
        List of positive integer lag orders (e.g., [1, 2, 3, 12]).

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['y_t', 'lag_1', 'lag_2', ...] sharing the original index.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 3: 12-Panel Lag Scatter Grid
lag_df = build_lag_matrix(retail_recent, lags=list(range(1, 13)))

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for idx, k in enumerate(range(1, 13)):
    ax = axes[idx // 4, idx % 4]
    valid = lag_df[['y_t', f'lag_{k}']].dropna()
    ax.scatter(valid[f'lag_{k}'], valid['y_t'], s=8, alpha=0.5, color='#2563eb')
    r = valid['y_t'].corr(valid[f'lag_{k}'])
    ax.set_title(f'Lag {k} (r = {r:.3f})', fontsize=9)
    ax.set_xlabel(f'$y_{{t-{k}}}$', fontsize=8)
    ax.set_ylabel('$y_t$', fontsize=8)
    ax.tick_params(labelsize=7)

fig.suptitle('US Retail Sales: Lag Scatter Grid (k = 1..12)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Part 4: Autocorrelation (ACF) & White Noise Testing (25 Minutes)

### The Sample Autocorrelation Function
$$r_k = \frac{\sum_{t=k+1}^{T}(y_t - \bar{y})(y_{t-k} - \bar{y})}{\sum_{t=1}^{T}(y_t - \bar{y})^2}$$

### Bartlett's 95% Confidence Bounds
Under the null hypothesis of white noise: $\pm \frac{1.96}{\sqrt{T}}$

### The Multi-Testing Trap & Ljung-Box
Testing 20 lags individually produces ~1 false positive. The Ljung-Box test aggregates:
$$Q^* = T(T+2) \sum_{k=1}^{h} \frac{r_k^2}{T-k} \sim \chi^2_h$$

### Three Canonical ACF Signatures
- **Trended macro series**: Positive, high, slowly decaying.
- **Seasonal series**: Scalloped sinusoidal with spikes at lags 12, 24.
- **White noise**: All spikes inside Bartlett bounds.

### Task 4a: `compute_sample_acf(series, max_lag=24) -> Dict[str, Any]`

#### 🛠️ Step-by-Step Logic to Write:
1. **Clean Array:** Drop `NaN`s from `series` and extract a 1D numpy float array: `y = series.dropna().values.astype(float)`.
2. **Compute Baseline Statistics:** Sample size $T = \text{len}(y)$, sample mean $\bar{y} = \text{np.mean}(y)$, and sum of squared deviations $\text{denom} = \sum_{t=1}^T (y_t - \bar{y})^2$.
3. **Loop Over Lags:** Initialize array `acf_vals = np.zeros(max_lag + 1)`.
   - For $k = 0$: $r_0 = 1.0$ (by definition, correlation of a variable with itself).
   - For $k \in 1 \dots \text{max\_lag}$: compute $\text{numer} = \sum_{t=k+1}^T (y_t - \bar{y})(y_{t-k} - \bar{y})$ using vector slicing `(y[k:] - y_bar) * (y[:T-k] - y_bar)`, and set $r_k = \text{numer} / \text{denom}$.
4. **Bartlett Threshold:** Compute `bartlett = 1.96 / np.sqrt(T)`.
5. **Return Dictionary:** Return keys `'lags'`, `'acf'`, `'bartlett_upper'`, `'bartlett_lower'`, and `'n_obs'`.

#### 🧰 Recommended Functions & Methods:
- `series.dropna().values.astype(float)`: Extracts clean 1D array of floats.
- `np.mean(y)` and `np.sum(...)`: Fast vectorized calculations.
- Vector slice alignment: `y[k:]` aligns $y_t$ with historical $y_{t-k}$ (`y[:T-k]`).
- `np.arange(max_lag + 1)`: Generates integer lags $[0, 1, \dots, \text{max\_lag}]$.
- `1.96 / np.sqrt(T)`: Bartlett 95% confidence threshold.

#### ⚠️ Critical Pitfalls to Avoid:
- **Denominator Convention:** In the sample ACF formula, always divide by $T$ (or $\sum(y_t - \bar{y})^2$) across all lags, NOT $T-k$. Dividing by $T-k$ can produce non-positive-definite covariance matrices.
- **Lag 0:** `acf[0]` must always equal exactly $1.0$.

---

### Task 4b: `test_white_noise(series, max_lag=12, alpha=0.05) -> Dict[str, Any]`

#### 🛠️ Step-by-Step Logic to Write:
1. **Clean Observations:** Clean the input series by removing missing values: `clean_s = series.dropna()`.
2. **Run Ljung-Box Test:** Import and execute `acorr_ljungbox(clean_s, lags=[max_lag], return_df=True)` from `statsmodels.tsa.stattools`.
3. **Extract Results:** Extract the $Q^*$ statistic (`lb_stat`) and asymptotic $p$-value (`lb_pvalue`) from the first row of the returned DataFrame.
4. **Decision Rule:** Evaluate the null hypothesis ($H_0$: white noise). If $p > \alpha$, the series is consistent with white noise (`is_white_noise = True`); if $p \le \alpha$, reject white noise (`is_white_noise = False`).
5. **Return Dictionary:** Return keys `'q_stat'` (float), `'p_value'` (float), and `'is_white_noise'` (bool).

#### 🧰 Recommended Functions & Methods:
- `from statsmodels.tsa.stattools import acorr_ljungbox`: Industry standard portmanteau test implementation.
- `acorr_ljungbox(series.dropna(), lags=[max_lag], return_df=True)`: Runs test at specified lag order.
- `float(res['lb_stat'].iloc[0])` and `float(res['lb_pvalue'].iloc[0])`: Safe scalar extraction.
- `bool(p_value > alpha)`: Boolean hypothesis test decision rule.

#### ⚠️ Critical Pitfalls to Avoid:
- **Hypothesis Interpretation:**
  - $H_0$: Series is independent white noise (no autocorrelation at lags $1 \dots h$).
  - $p > 0.05 \implies$ Fail to reject $H_0 \implies \textbf{White Noise} (`is_white_noise=True`).
  - $p \le 0.05 \implies$ Reject $H_0 \implies \textbf{Serial Dependence Present} (`is_white_noise=False`).

In [ ]:
def compute_sample_acf(
    series: pd.Series,
    max_lag: int = 24
) -> Dict[str, Any]:
    """
    Computes the sample Autocorrelation Function (ACF) and Bartlett confidence bounds.

    Parameters
    ----------
    series : pd.Series
        Input time series (NaNs will be dropped).
    max_lag : int, default=24
        Maximum lag order.

    Returns
    -------
    Dict[str, Any]
        'lags': np.ndarray [0..max_lag]
        'acf': np.ndarray of sample autocorrelations r_0..r_max
        'bartlett_upper': float (+1.96 / sqrt(T))
        'bartlett_lower': float (-1.96 / sqrt(T))
        'n_obs': int (sample size T)
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
def test_white_noise(
    series: pd.Series,
    max_lag: int = 12,
    alpha: float = 0.05
) -> Dict[str, Any]:
    """
    Executes the Ljung-Box portmanteau test for white noise.

    Parameters
    ----------
    series : pd.Series
        Input time series.
    max_lag : int, default=12
        Number of lags included in Q* statistic.
    alpha : float, default=0.05
        Significance level.

    Returns
    -------
    Dict[str, Any]
        'q_stat': float (Ljung-Box Q* statistic)
        'p_value': float
        'is_white_noise': bool (True if p_value > alpha)
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 4: Three-Panel ACF Comparison

# Generate white noise for contrast
np.random.seed(RANDOM_SEED)
white_noise = pd.Series(np.random.randn(240), index=pd.date_range('2000-01-01', periods=240, freq='MS'), name='white_noise')

series_dict = {
    'INDPRO (Trended)': indpro['2000':],
    'Retail Sales (Seasonal)': retail['2000':],
    'White Noise': white_noise
}

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (name, s) in zip(axes, series_dict.items()):
    acf_res = compute_sample_acf(s, max_lag=24)
    lb_res = test_white_noise(s, max_lag=12)
    
    ax.bar(acf_res['lags'][1:], acf_res['acf'][1:], width=0.6, color='#2563eb', alpha=0.7)
    ax.axhline(acf_res['bartlett_upper'], color='#dc2626', linestyle='--', lw=1, label='95% Bartlett')
    ax.axhline(acf_res['bartlett_lower'], color='#dc2626', linestyle='--', lw=1)
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_title(f'{name}\nLB p={lb_res["p_value"]:.4f} (WN={lb_res["is_white_noise"]})', fontsize=10)
    ax.set_xlabel('Lag $k$')
    ax.set_ylabel('$r_k$')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## Part 5: Interview Knowledge Checks & Repository Hygiene (20 Minutes)

Answer the 3 official syllabus practical questions below in the designated cells.

### Question 1
**How do you turn a pandas Series into a proper monthly datetime-indexed series, and what does `resample` do?**

*Write your answer here in markdown...*


### Question 2
**You pull a series from an API into pandas. What do you check before using it?**

*Write your answer here in markdown...*


### Question 3
**What belongs in a Git repository for a data project and what does not (data, keys, environment files)?**

*Write your answer here in markdown...*


## Summary & Bridge to Workshop 02

### What We Learned Today
1. **Temporal Data Integrity:** Clean `DatetimeIndex` with explicit frequency is the foundation of all time series analysis.
2. **Three Canonical Plots:** Time plots reveal the trajectory, seasonal plots reveal calendar patterns, subseries plots reveal structural month-level changes.
3. **Lag Geometry:** Serial dependence has visual geometry — persistence creates diagonals, seasonality creates clusters.
4. **Autocorrelation Summarizes Dependence:** The ACF $r_k$ and Ljung-Box $Q^*$ formalize what lag plots show geometrically.
5. **White Noise = Good Residuals:** A well-specified model leaves only white noise in its residuals.

### Coming Up in Workshop 02
**Transformations & Decomposition:** Log/Box-Cox transformations, classical vs. STL decomposition, and safe outlier handling.